# Generalizability Evaluation for Belief Tracking Research

This notebook evaluates whether the findings from the paper "Language Models use Lookbacks to Track Beliefs" (Prakash et al., 2025) generalize beyond the original experimental setting.

## Paper Summary

The paper investigates how language models (Llama-3-70B-Instruct and Llama-3.1-405B-Instruct) track beliefs of characters in Theory of Mind tasks. The key findings include:

1. **Answer Lookback Mechanism**: The answer payload (state token value) localizes to the final token residual stream after layer 56 with near-perfect IIA (Interchange Intervention Accuracy)
2. **Answer Pointer**: Answer pointer information is encoded at final token layers 34-52
3. **Binding Mechanism**: Binding address and payload occur between layers 33-38 at state token residual stream
4. **Source Reference**: Source reference (character and object OIs) encoded in layers 20-34

## Evaluation Checklist

| ID | Criterion | Description |
|----|-----------|-------------|
| GT1 | Model Generalization | Do the findings transfer to a new model not used in the original work? |
| GT2 | Data Generalization | Do the findings hold on new data instances not in the original dataset? |
| GT3 | Method Generalization | Can the method (causal abstraction for belief tracking) be applied to another similar task? |

In [1]:
# Setup and imports
import os
os.chdir('/home/smallyan/eval_agent')

import json
import random
import sys
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set the repository path
repo_path = '/net/scratch2/smallyan/belief_tracking_eval'
sys.path.insert(0, repo_path)

# Check CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

random.seed(42)

Using device: cuda
GPU: NVIDIA A100 80GB PCIe
Memory: 79.3 GB


In [2]:
# Import dataset utilities from the repo
from src.dataset import Sample, Dataset

# Load synthetic entities
data_path = os.path.join(repo_path, 'data', 'synthetic_entities')
with open(os.path.join(data_path, 'characters.json'), 'r') as f:
    all_characters = json.load(f)
with open(os.path.join(data_path, 'bottles.json'), 'r') as f:
    all_objects = json.load(f)
with open(os.path.join(data_path, 'drinks.json'), 'r') as f:
    all_states = json.load(f)

print(f"Loaded {len(all_characters)} characters, {len(all_objects)} objects, {len(all_states)} states")
print(f"Sample characters: {all_characters[:5]}")
print(f"Sample objects: {all_objects[:5]}")
print(f"Sample states: {all_states[:5]}")

Loaded 103 characters, 21 objects, 23 states
Sample characters: ['Dean', 'Beth', 'Jake', 'Josh', 'Karen']
Sample objects: ['jar', 'cup', 'mug', 'glass', 'flute']
Sample states: ['water', 'milk', 'tea', 'beer', 'soda']


## GT1: Model Generalization

**Objective**: Test whether the answer lookback mechanism (layer-specific IIA patterns) generalizes to a model NOT used in the original paper.

**Original models used**:
- Meta-Llama-3-70B-Instruct (primary)
- Meta-Llama-3.1-405B-Instruct (primary)
- Qwen2.5-14B-Instruct (extended testing)

**New model for testing**: Llama-3.1-8B-Instruct (smaller model from same family, but NOT explicitly tested in the paper)

**Hypothesis**: If the mechanism is generalizable, we should see similar layer-specific IIA patterns (high IIA at later layers for answer payload).

In [3]:
# Load Llama-3.1-8B-Instruct for GT1 evaluation
from nnsight import LanguageModel

print("Loading Llama-3.1-8B-Instruct...")
model_name = "meta-llama/Llama-3.1-8B-Instruct"
model = LanguageModel(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    dispatch=True,
)
print(f"Model loaded: {model_name}")
print(f"Number of layers: {model.config.num_hidden_layers}")

Loading Llama-3.1-8B-Instruct...


Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--meta-llama--Llama-3.1-8B-Instruct/.no_exist/0e9e39f249a16976918f6564b8830bc894c89659/adapter_config.json'


Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--meta-llama--Llama-3.1-8B-Instruct/.no_exist/0e9e39f249a16976918f6564b8830bc894c89659/adapter_config.json'


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded: meta-llama/Llama-3.1-8B-Instruct
Number of layers: 32


In [4]:
# Import utils from the causalToM_novis notebook
sys.path.insert(0, os.path.join(repo_path, 'notebooks', 'causalToM_novis'))
from utils import error_detection, get_reversed_sent_diff_state_counterfacts, get_answer_lookback_payload

print("Utilities imported successfully")

Utilities imported successfully


In [5]:
# GT1 Test 1: Basic model evaluation on CausalToM task
# First verify the model can do the task correctly

n_samples = 5
batch_size = 1

samples = []
for i in range(n_samples):
    characters = random.sample(all_characters, 2)
    objects = random.sample(all_objects, 2)
    states = random.sample(all_states, 2)
    samples.append(Sample(2, characters, objects, states))

dataset = Dataset(samples)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

print("Testing basic model performance on belief tracking task...")
correct, total = 0, 0
for bi, batch in enumerate(dataloader):
    prompt = batch["prompt"][0]
    target = batch["target"][0]
    
    with torch.no_grad():
        with model.trace(prompt):
            pred = model.lm_head.output[0, -1].argmax(dim=-1).save()
        
        pred_text = model.tokenizer.decode([pred]).lower().strip()
        if pred_text == target.lower().strip():
            correct += 1
        total += 1
        
        if bi == 0:  # Show first example
            print(f"\nExample prompt: {prompt[:200]}...")
            print(f"Target: {target}, Predicted: {pred_text}")

acc = correct / total
print(f"\nBasic accuracy: {acc:.2f} ({correct}/{total})")

Testing basic model performance on belief tracking task...


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Example prompt: Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A...
Target: unknown, Predicted: unknown



Basic accuracy: 0.60 (3/5)


In [6]:
# GT1 Test 2: Answer Lookback Payload - Test the layer-specific IIA pattern on new model
# The key finding is that answer payload (state token value) localizes to late layers

n_samples = 10
batch_size = 1

# Generate counterfactual dataset for answer lookback payload test
dataset_payload = get_answer_lookback_payload(
    all_characters,
    all_objects,
    all_states,
    n_samples,
)
dataloader_payload = DataLoader(dataset_payload, batch_size=batch_size, shuffle=False)

print(f"Created dataset with {len(dataset_payload)} samples for answer lookback payload test")

# Show an example
idx = 0
print("\nExample counterfactual pair:")
print("=" * 50)
print(f"Clean prompt: {dataset_payload[idx]['clean_prompt'][:300]}...")
print(f"Clean answer: {dataset_payload[idx]['clean_ans']}")
print(f"\nCounterfactual answer: {dataset_payload[idx]['counterfactual_ans']}")
print(f"Target (expected after intervention): {dataset_payload[idx]['target']}")

Created dataset with 10 samples for answer lookback payload test

Example counterfactual pair:
Clean prompt: Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe....
Clean answer: unknown

Counterfactual answer: beer
Target (expected after intervention): beer


In [7]:
# First, detect errors (samples where model doesn't answer correctly on both clean and counterfactual)
print("Detecting errors...")
_, errors = error_detection(model, dataloader_payload, is_remote=False)
print(f"Dataset size to be used for IIA: {len(dataset_payload) - len(errors)} ({len(errors)} errors)")

Detecting errors...


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:01<00:13,  1.50s/it]

 20%|██        | 2/10 [00:02<00:09,  1.18s/it]

 30%|███       | 3/10 [00:03<00:07,  1.09s/it]

...

 80%|████████  | 8/10 [00:08<00:02,  1.07s/it]

 90%|█████████ | 9/10 [00:09<00:01,  1.11s/it]

100%|██████████| 10/10 [00:11<00:00,  1.13s/it]

100%|██████████| 10/10 [00:11<00:00,  1.12s/it]

Dataset size to be used for IIA: 6 (4 errors)


In [8]:
# Run IIA experiment at key layers
# For 32-layer model, we expect effects in later layers (proportionally similar to 70B model)
# 70B has 80 layers, effect at 56+ = ~70% of layers
# 32 layers * 0.7 = ~22, so we expect effects around layer 22+

accs_answer_lookback_payload = {}
# Test a range of layers focusing on the later layers where we expect effects
patch_layers = [0, 8, 16, 20, 22, 24, 26, 28, 30, 31]

print("Testing IIA across layers for answer lookback payload...")
for layer_idx in patch_layers:
    correct, total = 0, 0
    for bi, batch in enumerate(dataloader_payload):
        if bi in errors:
            continue
        counterfactual_prompt = batch["counterfactual_prompt"][0]
        clean_prompt = batch["clean_prompt"][0]
        target = batch["target"][0]

        with torch.no_grad():
            with model.trace(counterfactual_prompt):
                counterfactual_layer_out = (
                    model.model.layers[layer_idx].output[0, -1].save()
                )

            with model.trace(clean_prompt):
                model.model.layers[layer_idx].output[0, -1] = counterfactual_layer_out
                pred = model.lm_head.output[0, -1].argmax(dim=-1).save()

            if model.tokenizer.decode([pred]).lower().strip() == target.lower().strip():
                correct += 1
            total += 1

            del pred
            torch.cuda.empty_cache()

    acc = round(correct / total, 3) if total > 0 else 0.0
    print(f"Layer {layer_idx:2d}: IIA = {acc:.3f} ({correct}/{total})")
    accs_answer_lookback_payload[layer_idx] = acc

print("\n=== GT1 Answer Lookback Payload Results ===")
print(f"Peak IIA: {max(accs_answer_lookback_payload.values()):.3f} at layer {max(accs_answer_lookback_payload, key=accs_answer_lookback_payload.get)}")

Testing IIA across layers for answer lookback payload...


TypeError: tuple indices must be integers or slices, not tuple

In [9]:
# Let's check how the layer output is structured
with model.trace("Hello world"):
    layer_out = model.model.layers[0].output.save()

print(f"Layer output type: {type(layer_out)}")
print(f"Layer output value type: {type(layer_out.value)}")
if isinstance(layer_out.value, tuple):
    print(f"Tuple length: {len(layer_out.value)}")
    print(f"First element shape: {layer_out.value[0].shape}")

Layer output type: <class 'nnsight.models.LanguageModel.LanguageModelProxy'>
Layer output value type: <class 'tuple'>
Tuple length: 2
First element shape: torch.Size([1, 3, 4096])


In [10]:
# The output is a tuple, need to access [0] first for hidden states
# Let's run the IIA experiment with the correct syntax

accs_answer_lookback_payload = {}
patch_layers = [0, 8, 16, 20, 22, 24, 26, 28, 30, 31]

print("Testing IIA across layers for answer lookback payload...")
for layer_idx in patch_layers:
    correct, total = 0, 0
    for bi, batch in enumerate(dataloader_payload):
        if bi in errors:
            continue
        counterfactual_prompt = batch["counterfactual_prompt"][0]
        clean_prompt = batch["clean_prompt"][0]
        target = batch["target"][0]

        with torch.no_grad():
            # First get the counterfactual layer output
            with model.trace(counterfactual_prompt):
                counterfactual_layer_out = model.model.layers[layer_idx].output[0][0, -1].save()
            
            # Then patch into clean and get prediction
            with model.trace(clean_prompt):
                model.model.layers[layer_idx].output[0][0, -1] = counterfactual_layer_out
                pred = model.lm_head.output[0, -1].argmax(dim=-1).save()

            if model.tokenizer.decode([pred]).lower().strip() == target.lower().strip():
                correct += 1
            total += 1

            del pred
            torch.cuda.empty_cache()

    acc = round(correct / total, 3) if total > 0 else 0.0
    print(f"Layer {layer_idx:2d}: IIA = {acc:.3f} ({correct}/{total})")
    accs_answer_lookback_payload[layer_idx] = acc

print("\n=== GT1 Answer Lookback Payload Results ===")
print(f"Peak IIA: {max(accs_answer_lookback_payload.values()):.3f} at layer {max(accs_answer_lookback_payload, key=accs_answer_lookback_payload.get)}")

Testing IIA across layers for answer lookback payload...


Layer  0: IIA = 0.000 (0/6)


Layer  8: IIA = 0.000 (0/6)


Layer 16: IIA = 0.167 (1/6)


Layer 20: IIA = 0.167 (1/6)


Layer 22: IIA = 0.167 (1/6)


Layer 24: IIA = 0.500 (3/6)


Layer 26: IIA = 0.833 (5/6)


Layer 28: IIA = 1.000 (6/6)
